# LeetCode 833 — Find And Replace in String bằng Python

Link đề: https://leetcode.com/problems/find-and-replace-in-string/ · Độ khó: **Medium** · Chủ đề: String, Array, Hash Table, Sorting (mô phỏng thay thế **đồng thời**)

Skeleton LeetCode:

```python
class Solution:
    def findReplaceString(self, s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
        ...
```

Ghi chú: cái bẫy của bài là chữ **"đồng thời (simultaneously)"** — mọi phép thay thế đều kiểm tra và áp dụng trên **chuỗi gốc**, không bị ảnh hưởng bởi vị trí dịch chuyển do các phép thay thế khác. Notebook đi từ cách thay từ phải sang trái, qua mảng đánh dấu, sắp xếp rồi ghép mảnh, và cách dùng độ lệch (offset).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `k` phép thay thế song song (`indices[i]`, `sources[i]`, `targets[i]`); phép `i` chỉ có tác dụng nếu `sources[i]` **xuất hiện đúng tại vị trí** `indices[i]` của **chuỗi gốc** `s`; khi đó thay đoạn đó bằng `targets[i]`; mọi phép áp dụng **đồng thời** và không chồng lấn.
2. Nắm các công cụ Python cần dùng: cắt chuỗi `s[i:j]`, `str.startswith(prefix, start)`, `zip`/`enumerate` cho các mảng song song, ghép chuỗi bằng danh sách và `"".join`, `sorted(..., key=...)`, tính **bất biến** của chuỗi.
3. Đi từ **thay từng phép từ phải sang trái** → **mảng đánh dấu + một lượt quét** → **sắp xếp theo chỉ số rồi ghép mảnh** → **dùng độ lệch (offset)**, và hiểu vì sao các cách cho cùng kết quả.
4. Hiểu mấu chốt: **mọi so khớp phải làm trên chuỗi gốc**; còn khi dựng kết quả phải **nhảy qua** `len(source)` ký tự gốc sau khi thay.
5. Tránh các lỗi kinh điển: thay lần lượt theo thứ tự đầu vào làm lệch chỉ số, quên nhảy qua phần `source`, dùng `s.replace` (thay mọi chỗ, bỏ qua chỉ số), kiểm tra bằng `src in s[i:]` thay vì `startswith` tại đúng vị trí, và quên sắp xếp `indices` khi ghép theo con trỏ.

## 2. Đề bài & ví dụ

Cho một chuỗi `s` (**đánh chỉ số từ 0**) và `k` phép thay thế, mô tả bằng ba mảng song song **đánh chỉ số từ 0**, cùng độ dài `k`: `indices`, `sources`, `targets`.

Để thực hiện phép thay thế thứ `i`:

1. Kiểm tra xem **chuỗi con** `sources[i]` có xuất hiện tại vị trí `indices[i]` trong **chuỗi gốc** `s` hay không.
2. Nếu **không** xuất hiện: **không làm gì**.
3. Nếu có: **thay** đoạn đó bằng `targets[i]`.

Ví dụ: `s = "abcd"`, `indices[i] = 0`, `sources[i] = "ab"`, `targets[i] = "eee"` thì kết quả là `"eeecd"`.

Mọi phép thay thế xảy ra **đồng thời**: chúng không ảnh hưởng chỉ số của nhau. Các test **đảm bảo các phép thay thế không chồng lấn** (ví dụ `s = "abc"`, `indices = [0, 1]`, `sources = ["ab", "bc"]` sẽ không xuất hiện vì `"ab"` và `"bc"` chồng nhau).

Trả về **chuỗi kết quả** sau khi thực hiện mọi phép thay thế.

**Ví dụ 1**

- Input: `s = "abcd", indices = [0, 2], sources = ["a", "cd"], targets = ["eee", "ffff"]`
- Output: `"eeebffff"`
- `"a"` xuất hiện ở chỉ số 0 nên thay bằng `"eee"`; `"cd"` xuất hiện ở chỉ số 2 nên thay bằng `"ffff"`.

**Ví dụ 2**

- Input: `s = "abcd", indices = [0, 2], sources = ["ab", "ec"], targets = ["eee", "ffff"]`
- Output: `"eeecd"`
- `"ab"` xuất hiện ở chỉ số 0 nên thay bằng `"eee"`; `"ec"` **không** xuất hiện ở chỉ số 2 nên bỏ qua.

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `k == indices.length == sources.length == targets.length`, `1 <= k <= 100`
- `0 <= indices[i] < s.length`
- `1 <= sources[i].length, targets[i].length <= 50`
- `s`, `sources[i]`, `targets[i]` chỉ gồm chữ cái thường.

Lưu ý: `indices` **không nhất thiết sắp xếp tăng dần**.

In [1]:
from typing import List
import random

examples = [
    ("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"], "eeebffff"),
    ("abcd", [0, 2], ["ab", "ec"], ["eee", "ffff"], "eeecd"),
]
for s, idx, src, tgt, expected in examples:
    print(f"s = {s!r}, indices = {idx}, sources = {src}, targets = {tgt} -> kỳ vọng {expected!r}")

s = 'abcd', indices = [0, 2], sources = ['a', 'cd'], targets = ['eee', 'ffff'] -> kỳ vọng 'eeebffff'
s = 'abcd', indices = [0, 2], sources = ['ab', 'ec'], targets = ['eee', 'ffff'] -> kỳ vọng 'eeecd'


## 3. Kiến thức nền

### 3.1. Cắt chuỗi và `startswith(prefix, start)`

`s[i:j]` lấy các ký tự từ chỉ số `i` đến `j - 1` (bỏ ký tự ở `j`). Nếu `j` vượt độ dài chuỗi thì chỉ lấy đến hết chuỗi (không báo lỗi). Do đó "đoạn `src` có xuất hiện **đúng tại** vị trí `i` không?" có thể viết `s[i:i + len(src)] == src`.

Python còn có `s.startswith(src, i)`: kiểm tra `s[i:]` có **bắt đầu bằng** `src` không — đúng điều ta cần, ngắn gọn hơn, và an toàn khi `src` dài hơn phần còn lại của `s` (kết quả `False`).

In [2]:
s = "abcd"
print(s[0:2], s[2:4], s[2:100], repr(s[4:6]))      # ab cd cd ''
print(s[2:2 + len("cd")] == "cd")                   # True
print(s.startswith("cd", 2), s.startswith("ec", 2), s.startswith("cde", 2))   # True False False
print("cd" in s[0:])                                 # True: 'cd' có trong chuỗi, nhưng KHÔNG chứng tỏ nó ở vị trí 0!
print(s.startswith("cd", 0))                         # False: đúng vị trí mới tính

ab cd cd ''
True
True False False
True
False


### 3.2. Các mảng song song: `zip` và `enumerate`

Đề cho ba danh sách cùng độ dài; phần tử cùng vị trí thuộc **cùng một phép thay thế**. `zip(indices, sources, targets)` ghép chúng thành các bộ ba; `enumerate` thêm số thứ tự `k`.

In [3]:
indices = [0, 2]
sources = ["a", "cd"]
targets = ["eee", "ffff"]
for k, (i, src, tgt) in enumerate(zip(indices, sources, targets)):
    print(f"phép {k}: tại {i}, tìm {src!r}, thay bằng {tgt!r}")
ops = list(zip(indices, sources, targets))
print(ops)

phép 0: tại 0, tìm 'a', thay bằng 'eee'
phép 1: tại 2, tìm 'cd', thay bằng 'ffff'
[(0, 'a', 'eee'), (2, 'cd', 'ffff')]


### 3.3. Chuỗi bất biến: dựng kết quả bằng danh sách + `"".join`

Chuỗi trong Python **không sửa được tại chỗ**; mỗi lần `s = s[:i] + t + s[j:]` tạo một chuỗi **mới** (tốn `O(n)`). Khi cần ghép nhiều mảnh, ta gom các mảnh vào danh sách `parts` rồi `"".join(parts)` một lần (tổng `O(độ dài)`), thay vì nối `+=` nhiều lần.

In [4]:
parts = []
parts.append("eee")
parts.append("b")
parts.append("ffff")
print("".join(parts))                      # eeebffff

s = "abcd"
t = s[:2] + "XY" + s[4:]                   # thay s[2:4] bằng "XY": phép thay đoạn tại chỗ
print(t, "| s vẫn là", s)

eeebffff
abXY | s vẫn là abcd


### 3.4. `sorted` với `key` (sắp xếp theo chỉ số)

`sorted(ops, key=lambda op: op[0])` sắp xếp các bộ ba theo **phần tử đầu** (chỉ số). `sorted` là sắp xếp **ổn định**: hai phần tử cùng khoá giữ nguyên thứ tự ban đầu. Muốn sắp giảm dần dùng `reverse=True`. Để sắp xếp **các chỉ số của phép thay thế** mà không mất mối liên hệ ba mảng, ta sắp xếp danh sách các bộ ba (hoặc danh sách các số thứ tự phép).

In [5]:
ops = [(2, "cd", "ffff"), (0, "a", "eee")]            # chỉ số không theo thứ tự
print(sorted(ops, key=lambda op: op[0]))
print(sorted(ops, key=lambda op: op[0], reverse=True))
order = sorted(range(len(ops)), key=lambda j: ops[j][0])
print(order)                                          # [1, 0]: phép số 1 có chỉ số nhỏ nhất

[(0, 'a', 'eee'), (2, 'cd', 'ffff')]
[(2, 'cd', 'ffff'), (0, 'a', 'eee')]
[1, 0]


### 3.5. "Đồng thời" nghĩa là gì: so khớp trên **chuỗi gốc**

Nếu ta thay lần lượt rồi **kiểm tra trên chuỗi đã bị sửa**, các chỉ số bị lệch: sau khi thay `"a"` (độ dài 1) bằng `"eee"` (độ dài 3), phần sau dịch sang phải 2 vị trí, và `indices[1] = 2` không còn trỏ vào `"cd"`.

Đồng thời nghĩa là: mọi so khớp so với **`s` gốc**; kết quả cuối cùng là ghép các đoạn: *đoạn không đụng tới* + *`target`* + ... theo thứ tự xuất hiện trong `s` gốc.

In [6]:
s = "abcd"
# thay lần lượt trên chuỗi đã sửa (SAI so với đồng thời)
cur = s
cur = cur[:0] + "eee" + cur[1:]                 # phép 'a' -> 'eee'
print(cur, "| cur.startswith('cd', 2)?", cur.startswith("cd", 2))   # eeebcd | False: lệch chỉ số
print("trên chuỗi gốc:", s.startswith("cd", 2))                   # True

eeebcd | cur.startswith('cd', 2)? False
trên chuỗi gốc: True


## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 1: `s = "abcd"`.

- Phép 0: tại chỉ số 0, `"a"` có khớp? `s[0:1] = "a"` ✓ → đánh dấu "tại 0 thay `"a"` bằng `"eee"`".
- Phép 1: tại chỉ số 2, `"cd"` có khớp? `s[2:4] = "cd"` ✓ → đánh dấu "tại 2 thay `"cd"` bằng `"ffff"`".

Dựng kết quả bằng cách đi qua `s` gốc từ trái sang:

- chỉ số 0: có phép → ghi `"eee"`, nhảy qua `len("a") = 1` ký tự → tới chỉ số 1;
- chỉ số 1: không phép → ghi `s[1] = "b"`, sang 2;
- chỉ số 2: có phép → ghi `"ffff"`, nhảy qua `len("cd") = 2` → tới 4: hết.

Kết quả `"eee" + "b" + "ffff" = "eeebffff"` ✓.

Ví dụ 2: phép 1 tại chỉ số 2 với `"ec"`: `s[2:4] = "cd" ≠ "ec"` ✗ → bỏ qua. Kết quả: `"eee"` (thay `"ab"`) + `"cd"` = `"eeecd"` ✓.

Ý tưởng chung: **(1) kiểm tra mọi phép trên chuỗi gốc để biết phép nào có hiệu lực; (2) dựng chuỗi mới từ trái sang phải**.

In [7]:
s = "abcd"
ops = [(0, "a", "eee"), (2, "cd", "ffff")]
for i, src, tgt in ops:
    print(f"tại {i}: s[{i}:{i + len(src)}] = {s[i:i + len(src)]!r} {'==' if s[i:i + len(src)] == src else '!='} {src!r}")

tại 0: s[0:1] = 'a' == 'a'
tại 2: s[2:4] = 'cd' == 'cd'


## 5. Cách 1 — Brute Force: thay từng phép từ phải sang trái trên chuỗi

### Ý tưởng

Vấn đề của "thay lần lượt" là các thay đổi ở **bên trái** làm dịch chỉ số **bên phải**. Nếu ta xử lý các phép theo thứ tự **chỉ số giảm dần** (từ phải sang trái) thì khi thay tại `i`, mọi thay đổi trước đó đều ở **bên phải** `i` — **không ảnh hưởng** đoạn `[i, i + len(src))` và các chỉ số nhỏ hơn. Do đó ta có thể sửa trực tiếp chuỗi hiện tại mà vẫn so khớp đúng như trên chuỗi gốc.

1. Sắp xếp các phép theo `indices` giảm dần.
2. Với mỗi phép `(i, src, tgt)`: nếu `res.startswith(src, i)` thì `res = res[:i] + tgt + res[i + len(src):]`.

(Điều kiện không chồng lấn đảm bảo đoạn `[i, i + len(src))` chưa bị các phép bên phải động tới.)

### Code

In [8]:
def find_replace_right_to_left(s, indices, sources, targets):
    order = sorted(range(len(indices)), key=lambda j: indices[j], reverse=True)
    res = s
    for j in order:
        i, src, tgt = indices[j], sources[j], targets[j]
        if res.startswith(src, i):                 # phần bên trái i vẫn như chuỗi gốc
            res = res[:i] + tgt + res[i + len(src):]
    return res

for s, idx, src, tgt, expected in examples:
    got = find_replace_right_to_left(s, idx, src, tgt)
    print(got, expected, "OK" if got == expected else "SAI")

eeebffff eeebffff OK
eeecd eeecd OK


### Độ phức tạp và nút thắt

- **Time:** `O(k log k + k · n)` — sắp xếp `O(k log k)`; mỗi phép thay tạo chuỗi mới `O(n)`; với `k ≤ 100, n ≤ 1000` (chuỗi có thể dài thêm tới `50·100 = 5000`) vẫn nhỏ.
- **Space:** `O(n + L)` cho chuỗi kết quả (mỗi lần sinh một bản mới).

**Nút thắt:** ta dựng lại **toàn bộ chuỗi** mỗi lần thay. Có thể dựng kết quả **một lần**: đánh dấu phép nào hiệu lực, rồi quét `s` một lượt (Cách 2). Cách 1 vẫn hữu ích vì rất ngắn và là điểm xuất phát tự nhiên của ý "tránh dịch chỉ số".

## 6. Ý tưởng thuật toán / bước đột phá

Tách bài toán làm **hai pha độc lập**:

1. **Pha kiểm tra:** với mỗi phép `i`, so khớp `sources[i]` với `s` gốc tại `indices[i]`. Pha này **chỉ đọc** `s` nên không ai ảnh hưởng ai (đúng nghĩa "đồng thời"). Ghi nhận kết quả vào một mảng `d` độ dài `n`: `d[indices[i]] = i` nếu khớp, `-1` nếu chưa/không có.
2. **Pha dựng:** duyệt `s` gốc từ trái sang bằng chỉ số `p`:
   - nếu `d[p] != -1`: thêm `targets[d[p]]` vào kết quả, rồi `p += len(sources[d[p]])` (**nhảy qua** đoạn nguồn);
   - ngược lại thêm `s[p]` và `p += 1`.

Vì các phép không chồng lấn, mỗi ký tự của `s` hoặc thuộc **đúng một** đoạn nguồn có hiệu lực (bị thay) hoặc không (giữ nguyên). Dựng một lần: `O(n + L)` với `L` là tổng độ dài `targets`.

Cell dưới in mảng `d` của Ví dụ 1 và Ví dụ 2 để thấy sự khác biệt giữa "có hiệu lực" và "không".

In [9]:
def build_marker(s, indices, sources):
    d = [-1] * len(s)
    for k, (i, src) in enumerate(zip(indices, sources)):
        if s.startswith(src, i):
            d[i] = k
    return d

for s, idx, src, tgt, expected in examples:
    print(f"s = {s!r}: d = {build_marker(s, idx, src)}")

s = 'abcd': d = [0, -1, 1, -1]
s = 'abcd': d = [0, -1, -1, -1]


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mảng đánh dấu + một lượt quét (lời giải nộp)

**Giải thích từng đoạn.**

1. `d = [-1] * n`.
2. Với mỗi phép `k`: nếu `s.startswith(sources[k], indices[k])` thì `d[indices[k]] = k`.
3. Duyệt `p` từ `0`: nếu `d[p] != -1` thì ghi `targets[d[p]]` và `p += len(sources[d[p]])`; ngược lại ghi `s[p]` và `p += 1`.
4. `"".join(parts)`.

**Code demo:**

In [10]:
def find_replace_marker(s, indices, sources, targets):
    n = len(s)
    d = [-1] * n
    for k, (i, src) in enumerate(zip(indices, sources)):
        if s.startswith(src, i):
            d[i] = k
    parts = []
    p = 0
    while p < n:
        if d[p] != -1:
            k = d[p]
            parts.append(targets[k])
            p += len(sources[k])               # nhảy qua đoạn nguồn
        else:
            parts.append(s[p])
            p += 1
    return "".join(parts)

for s, idx, src, tgt, expected in examples:
    got = find_replace_marker(s, idx, src, tgt)
    print(got, expected, "OK" if got == expected else "SAI")

eeebffff eeebffff OK
eeecd eeecd OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionMarker:
    def findReplaceString(self, s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
        n = len(s)
        d = [-1] * n
        for k, (i, src) in enumerate(zip(indices, sources)):
            if s.startswith(src, i):
                d[i] = k
        ans = []
        p = 0
        while p < n:
            if d[p] != -1:
                ans.append(targets[d[p]])
                p += len(sources[d[p]])
            else:
                ans.append(s[p])
                p += 1
        return "".join(ans)

print(SolutionMarker().findReplaceString("abcd", [0, 2], ["ab", "ec"], ["eee", "ffff"]))   # eeecd

eeecd


**Độ phức tạp (Cách 2):** Time `O(n + L)` với `L` là tổng độ dài của `sources` (để so khớp) và `targets` (để ghép) — mỗi phép so khớp tốn `O(|source|)`. Space `O(n + L)`.

### 7.2. Cách 3 — Sắp xếp theo chỉ số rồi ghép mảnh bằng con trỏ

**Giải thích từng đoạn.**

1. Gom các phép thành bộ ba rồi **sắp xếp theo chỉ số tăng dần**.
2. Giữ con trỏ `prev = 0` (đã ghép xong `s[:prev]`).
3. Với mỗi phép `(i, src, tgt)` theo thứ tự: nếu `s.startswith(src, i)`: ghép `s[prev:i]` (đoạn không bị đụng) rồi `tgt`, và đặt `prev = i + len(src)`.
4. Cuối cùng ghép nốt `s[prev:]`.

Không cần mảng `d` dài `n`; chỉ xử lý `k` phép. Thời gian `O(k log k + n + L)`.

**Code demo:**

In [12]:
def find_replace_sorted(s, indices, sources, targets):
    ops = sorted(zip(indices, sources, targets), key=lambda op: op[0])
    parts = []
    prev = 0
    for i, src, tgt in ops:
        if s.startswith(src, i):
            parts.append(s[prev:i])         # đoạn nằm giữa phép trước và phép này
            parts.append(tgt)
            prev = i + len(src)
    parts.append(s[prev:])
    return "".join(parts)

for s, idx, src, tgt, expected in examples:
    got = find_replace_sorted(s, idx, src, tgt)
    print(got, expected, "OK" if got == expected else "SAI")
# chỉ số KHÔNG sắp xếp sẵn: hàm vẫn đúng vì ta tự sắp xếp
print(find_replace_sorted("abcd", [2, 0], ["cd", "a"], ["ffff", "eee"]))   # eeebffff

eeebffff eeebffff OK
eeecd eeecd OK
eeebffff


**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionSorted:
    def findReplaceString(self, s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
        ops = sorted(zip(indices, sources, targets), key=lambda op: op[0])
        parts = []
        prev = 0
        for i, src, tgt in ops:
            if s.startswith(src, i):
                parts.append(s[prev:i])
                parts.append(tgt)
                prev = i + len(src)
        parts.append(s[prev:])
        return "".join(parts)

print(SolutionSorted().findReplaceString("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))   # eeebffff

eeebffff


**Độ phức tạp (Cách 3):** Time `O(k log k + n + L)`, Space `O(n + L)`.

### 7.3. Cách 4 — Sắp xếp + độ lệch (offset) trên chuỗi đang sửa

**Giải thích từng đoạn.** Một cách "thủ công" nhưng thường gặp trong thực tế:

1. Sắp xếp các phép theo chỉ số tăng dần.
2. Giữ `offset` = tổng `len(target) − len(source)` của các phép **đã áp dụng** (độ dịch chuyển của phần bên phải).
3. Với mỗi phép: **so khớp trên `s` gốc** (`s.startswith(src, i)`); nếu khớp, vị trí trong chuỗi đang sửa là `i + offset`; thay đoạn đó: `res = res[:i + offset] + tgt + res[i + offset + len(src):]`; rồi `offset += len(tgt) − len(src)`.

Ý chính: ta **vẫn kiểm tra trên chuỗi gốc**, chỉ **ánh xạ chỉ số** sang chuỗi đang sửa bằng `offset`.

**Code demo:**

In [14]:
def find_replace_offset(s, indices, sources, targets):
    ops = sorted(zip(indices, sources, targets), key=lambda op: op[0])
    res = s
    offset = 0
    for i, src, tgt in ops:
        if s.startswith(src, i):                   # so khớp trên chuỗi GỐC
            pos = i + offset                       # vị trí tương ứng trong chuỗi đang sửa
            res = res[:pos] + tgt + res[pos + len(src):]
            offset += len(tgt) - len(src)
    return res

for s, idx, src, tgt, expected in examples:
    got = find_replace_offset(s, idx, src, tgt)
    print(got, expected, "OK" if got == expected else "SAI")

eeebffff eeebffff OK
eeecd eeecd OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionOffset:
    def findReplaceString(self, s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
        res = s
        offset = 0
        for i, src, tgt in sorted(zip(indices, sources, targets), key=lambda op: op[0]):
            if s.startswith(src, i):
                pos = i + offset
                res = res[:pos] + tgt + res[pos + len(src):]
                offset += len(tgt) - len(src)
        return res

print(SolutionOffset().findReplaceString("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))   # eeebffff

eeebffff


**Độ phức tạp (Cách 4):** Time `O(k log k + k · (n + L))` (mỗi phép thay tạo một chuỗi mới `O(n + L)`), Space `O(n + L)`. Chậm hơn Cách 2 và 3 về lý thuyết nhưng với `k ≤ 100` không đáng kể.

## 8. TRACE từng bước

Trace Cách 2 (mảng đánh dấu) trên Ví dụ 1: `s = "abcd"`, phép 0 `(0, "a", "eee")`, phép 1 `(2, "cd", "ffff")`.

Pha kiểm tra: `s.startswith("a", 0)` ✓ → `d[0] = 0`; `s.startswith("cd", 2)` ✓ → `d[2] = 1`. Vậy `d = [0, -1, 1, -1]`.

Pha dựng:

| p | `d[p]` | hành động | `parts` sau bước | p mới |
|---|---|---|---|---|
| 0 | 0 | ghi `"eee"`, nhảy `len("a") = 1` | `["eee"]` | 1 |
| 1 | -1 | ghi `s[1] = "b"` | `["eee", "b"]` | 2 |
| 2 | 1 | ghi `"ffff"`, nhảy `len("cd") = 2` | `["eee", "b", "ffff"]` | 4 |

Hết (`p = 4 = n`): `"eeebffff"`. Hàm `trace_marker` in các bước này cho input bất kỳ.

In [16]:
def trace_marker(s, indices, sources, targets):
    n = len(s)
    print(f"s = {s!r}")
    d = [-1] * n
    print("-- Pha 1: kiểm tra trên chuỗi gốc --")
    for k, (i, src) in enumerate(zip(indices, sources)):
        ok = s.startswith(src, i)
        print(f"  phép {k}: tại {i}, nguồn {src!r}: s[{i}:{i + len(src)}] = {s[i:i + len(src)]!r} -> {'KHỚP' if ok else 'không khớp'}")
        if ok:
            d[i] = k
    print("  d =", d)
    print("-- Pha 2: dựng kết quả --")
    parts = []
    p = 0
    while p < n:
        if d[p] != -1:
            k = d[p]
            parts.append(targets[k])
            print(f"  p={p}: d[p]={k} -> ghi {targets[k]!r}, nhảy {len(sources[k])} -> p={p + len(sources[k])}")
            p += len(sources[k])
        else:
            parts.append(s[p])
            print(f"  p={p}: không có phép -> ghi {s[p]!r}, p={p + 1}")
            p += 1
    result = "".join(parts)
    print("  kết quả =", repr(result))
    return result

trace_marker("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"])
print()
trace_marker("abcd", [0, 2], ["ab", "ec"], ["eee", "ffff"])

s = 'abcd'
-- Pha 1: kiểm tra trên chuỗi gốc --
  phép 0: tại 0, nguồn 'a': s[0:1] = 'a' -> KHỚP
  phép 1: tại 2, nguồn 'cd': s[2:4] = 'cd' -> KHỚP
  d = [0, -1, 1, -1]
-- Pha 2: dựng kết quả --
  p=0: d[p]=0 -> ghi 'eee', nhảy 1 -> p=1
  p=1: không có phép -> ghi 'b', p=2
  p=2: d[p]=1 -> ghi 'ffff', nhảy 2 -> p=4
  kết quả = 'eeebffff'

s = 'abcd'
-- Pha 1: kiểm tra trên chuỗi gốc --
  phép 0: tại 0, nguồn 'ab': s[0:2] = 'ab' -> KHỚP
  phép 1: tại 2, nguồn 'ec': s[2:4] = 'cd' -> không khớp
  d = [0, -1, -1, -1]
-- Pha 2: dựng kết quả --
  p=0: d[p]=0 -> ghi 'eee', nhảy 2 -> p=2
  p=2: không có phép -> ghi 'c', p=3
  p=3: không có phép -> ghi 'd', p=4
  kết quả = 'eeecd'


'eeecd'

Trace thứ hai: Cách 3 (sắp xếp + con trỏ `prev`) trên một ví dụ có chỉ số **không sắp xếp** và độ dài `target` khác `source`: `s = "abcdefgh"`, các phép `(5, "fg", "X")`, `(1, "bc", "YYY")`, `(3, "zz", "Q")` (phép này không khớp).

In [17]:
def trace_sorted(s, indices, sources, targets):
    ops = sorted(zip(indices, sources, targets), key=lambda op: op[0])
    print("sau khi sắp xếp theo chỉ số:", ops)
    parts = []
    prev = 0
    for i, src, tgt in ops:
        ok = s.startswith(src, i)
        if ok:
            parts.append(s[prev:i])
            parts.append(tgt)
            print(f"  phép ({i}, {src!r}, {tgt!r}) KHỚP: ghép s[{prev}:{i}] = {s[prev:i]!r} rồi {tgt!r}; prev {prev} -> {i + len(src)}")
            prev = i + len(src)
        else:
            print(f"  phép ({i}, {src!r}, {tgt!r}) không khớp: bỏ qua")
    parts.append(s[prev:])
    print(f"  ghép nốt s[{prev}:] = {s[prev:]!r}")
    result = "".join(parts)
    print("  kết quả =", repr(result))
    return result

trace_sorted("abcdefgh", [5, 1, 3], ["fg", "bc", "zz"], ["X", "YYY", "Q"])

sau khi sắp xếp theo chỉ số: [(1, 'bc', 'YYY'), (3, 'zz', 'Q'), (5, 'fg', 'X')]
  phép (1, 'bc', 'YYY') KHỚP: ghép s[0:1] = 'a' rồi 'YYY'; prev 0 -> 3
  phép (3, 'zz', 'Q') không khớp: bỏ qua
  phép (5, 'fg', 'X') KHỚP: ghép s[3:5] = 'de' rồi 'X'; prev 3 -> 7
  ghép nốt s[7:] = 'h'
  kết quả = 'aYYYdeXh'


'aYYYdeXh'

Trace thứ ba: Cách 4 (độ lệch `offset`) cho cùng ví dụ, để thấy chuỗi đang sửa và `offset` thay đổi thế nào.

In [18]:
def trace_offset(s, indices, sources, targets):
    res = s
    offset = 0
    print("chuỗi gốc:", s)
    for i, src, tgt in sorted(zip(indices, sources, targets), key=lambda op: op[0]):
        if s.startswith(src, i):
            pos = i + offset
            before = res
            res = res[:pos] + tgt + res[pos + len(src):]
            offset += len(tgt) - len(src)
            print(f"  ({i}, {src!r}, {tgt!r}): pos = {i} + {offset - (len(tgt) - len(src))} = {pos}; {before!r} -> {res!r}; offset = {offset}")
        else:
            print(f"  ({i}, {src!r}, {tgt!r}): không khớp trên chuỗi gốc, bỏ qua")
    return res

print(trace_offset("abcdefgh", [5, 1, 3], ["fg", "bc", "zz"], ["X", "YYY", "Q"]))

chuỗi gốc: abcdefgh
  (1, 'bc', 'YYY'): pos = 1 + 0 = 1; 'abcdefgh' -> 'aYYYdefgh'; offset = 1
  (3, 'zz', 'Q'): không khớp trên chuỗi gốc, bỏ qua
  (5, 'fg', 'X'): pos = 5 + 1 = 6; 'aYYYdefgh' -> 'aYYYdeXh'; offset = 0
aYYYdeXh


## 9. Code nộp LeetCode

Lời giải chuẩn: mảng đánh dấu + một lượt quét (Cách 2) — dễ chứng minh, `O(n + L)`, thể hiện đúng tinh thần "đồng thời". Đúng `class Solution` và signature.

In [19]:
from typing import List

class Solution:
    def findReplaceString(self, s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
        n = len(s)
        d = [-1] * n                                   # d[i] = số thứ tự phép có hiệu lực bắt đầu tại i
        for k, (i, src) in enumerate(zip(indices, sources)):
            if s.startswith(src, i):                   # so khớp trên chuỗi GỐC
                d[i] = k
        ans = []
        p = 0
        while p < n:
            if d[p] != -1:
                ans.append(targets[d[p]])              # ghi target
                p += len(sources[d[p]])                # nhảy qua đoạn nguồn trong chuỗi gốc
            else:
                ans.append(s[p])
                p += 1
        return "".join(ans)

sol = Solution()
for s, idx, src, tgt, expected in examples:
    got = sol.findReplaceString(s, idx, src, tgt)
    print(repr(got), repr(expected), "OK" if got == expected else "SAI")

'eeebffff' 'eeebffff' OK
'eeecd' 'eeecd' OK


## 10. Vì sao đúng

Ta chứng minh cho Cách 2 (các cách khác tương đương) với giả thiết đề: **các đoạn nguồn có hiệu lực không chồng lấn**.

**Định nghĩa.** Gọi `A` là tập các phép `k` có `s.startswith(sources[k], indices[k])` (có hiệu lực). Mỗi phép `k ∈ A` chiếm đoạn `I_k = [indices[k], indices[k] + len(sources[k]))` của `s` gốc. Theo đề các `I_k` đôi một rời nhau. Kết quả mong muốn: thay mỗi đoạn `I_k` (`k ∈ A`) bằng `targets[k]`, giữ nguyên các ký tự ngoài mọi đoạn, theo thứ tự trái → phải của `s` gốc.

**Quy nạp theo vị trí `p`** (bất biến vòng `while`): tại mỗi lần bắt đầu vòng, `parts` đã chứa đúng kết quả mong muốn cho `s[:p]`, và `p` không nằm ở **giữa** một đoạn `I_k`.

- Khởi đầu: `p = 0`, `parts` rỗng ✓.
- Nếu `d[p] = k ≠ -1`: `p` là đầu đoạn `I_k`. Ghi `targets[k]` và nhảy tới cuối đoạn (`p + len(sources[k])`): đúng kết quả cho `s[:p + len]`, và vị trí mới không nằm giữa đoạn nào do các đoạn rời nhau.
- Nếu `d[p] = -1`: vì bất biến, `p` không nằm giữa đoạn nào và cũng không là đầu đoạn nào, nên `s[p]` nằm ngoài mọi đoạn: ghi nguyên `s[p]`, `p += 1` ✓.

Hết vòng, `p = n`: `parts` là kết quả mong muốn.

**Vì sao kiểm tra trên chuỗi gốc là đúng:** đề nói thay thế **đồng thời** và `indices[i]` là chỉ số trong **chuỗi gốc**. Pha kiểm tra chỉ đọc `s`, nên không bị ảnh hưởng bởi phép khác.

**Trường hợp hai phép cùng chỉ số:** nếu `indices[a] = indices[b]` thì hai đoạn nguồn bắt đầu cùng chỗ nên chồng nhau nếu cả hai khớp; đề cấm. Nếu chỉ một cái khớp thì `d[i]` ghi đúng phép đó (phép không khớp không ghi gì).

Cell dưới kiểm chứng bằng **oracle độc lập theo từng "ô" của chuỗi gốc**: tạo danh sách `slots` với `slots[j]` ban đầu là `s[j]`; với mỗi phép có hiệu lực, đặt `slots[i] = target` và xoá các ô `i+1 ... i+len-1` (`""`); ghép tất cả `slots`. Oracle này không dùng con trỏ nhảy hay dịch chỉ số.

In [20]:
def oracle_slots(s, indices, sources, targets):
    slots = list(s)
    for i, src, tgt in zip(indices, sources, targets):
        if s[i:i + len(src)] == src:               # so sánh bằng cắt chuỗi, không dùng startswith
            slots[i] = tgt
            for j in range(i + 1, i + len(src)):
                slots[j] = ""
    return "".join(slots)

for s, idx, src, tgt, expected in examples:
    assert oracle_slots(s, idx, src, tgt) == expected
print("oracle khớp hai ví dụ đề")

def random_case(rng, max_n=60, alphabet="ab", match_prob=0.6):
    """Sinh test hợp lệ: các đoạn nguồn (kể cả không khớp) đôi một rời nhau; các phép đảo thứ tự ngẫu nhiên."""
    n = rng.randint(1, max_n)
    s = "".join(rng.choice(alphabet) for _ in range(n))
    ops = []
    cursor = 0
    while cursor < n and len(ops) < 100:
        start = rng.randint(cursor, min(n - 1, cursor + rng.randint(0, 5)))
        length = rng.randint(1, min(50, n - start, 4))
        if rng.random() < match_prob:
            src = s[start:start + length]
        else:
            src = "".join(rng.choice(alphabet + "z") for _ in range(length))
        tgt = "".join(rng.choice(alphabet + "xy") for _ in range(rng.randint(1, 6)))
        ops.append((start, src, tgt))
        cursor = start + length
        if rng.random() < 0.15:
            break
    rng.shuffle(ops)
    indices = [o[0] for o in ops]
    sources = [o[1] for o in ops]
    targets = [o[2] for o in ops]
    return s, indices, sources, targets

rng = random.Random(833)
for _ in range(3000):
    s, idx, src, tgt = random_case(rng)
    expected = oracle_slots(s, idx, src, tgt)
    assert Solution().findReplaceString(s, idx, src, tgt) == expected, (s, idx, src, tgt)
print("3000 input ngẫu nhiên (chỉ số không sắp xếp): Solution khớp oracle theo ô")

oracle khớp hai ví dụ đề


3000 input ngẫu nhiên (chỉ số không sắp xếp): Solution khớp oracle theo ô


## 11. Time / Space Complexity

Gọi `n = len(s)`, `k` số phép, `L_s` tổng độ dài `sources`, `L_t` tổng độ dài `targets`.

- **Time:** `O(n + L_s + L_t)` — pha kiểm tra tốn `O(L_s)` (mỗi `startswith` so tối đa `len(src)` ký tự), pha dựng tốn `O(n + L_t)`.
- **Space:** `O(n + L_t)` — mảng `d` dài `n` và danh sách mảnh.

Với giới hạn `n ≤ 1000`, `k ≤ 100`, độ dài mỗi source/target ≤ 50 thì `L_s, L_t ≤ 5000`: các thao tác chỉ vài nghìn bước.

Cell dưới so sánh bốn cách trên input lớn nhất theo đề (n = 1000, k = 100): kết quả giống nhau.

In [21]:
def max_case(rng):
    n = 1000
    s = "".join(rng.choice("abc") for _ in range(n))
    ops = []
    for t in range(100):
        start = t * 10
        length = rng.randint(1, 8)
        src = s[start:start + length] if rng.random() < 0.7 else "z" * length
        tgt = "".join(rng.choice("xyz") for _ in range(rng.randint(1, 50)))
        ops.append((start, src, tgt))
    rng.shuffle(ops)
    return s, [o[0] for o in ops], [o[1] for o in ops], [o[2] for o in ops]

rng = random.Random(11)
s, idx, src, tgt = max_case(rng)
results = {
    "right_to_left": find_replace_right_to_left(s, idx, src, tgt),
    "marker": find_replace_marker(s, idx, src, tgt),
    "sorted": find_replace_sorted(s, idx, src, tgt),
    "offset": find_replace_offset(s, idx, src, tgt),
    "oracle": oracle_slots(s, idx, src, tgt),
}
assert len(set(results.values())) == 1
print("n = 1000, k = 100: mọi cách cho cùng kết quả, độ dài kết quả =", len(results["marker"]))

n = 1000, k = 100: mọi cách cho cùng kết quả, độ dài kết quả = 2245


## 12. Bộ test

Gồm: ví dụ đề, edge case, phản ví dụ cho các lỗi, đối chiếu các cách bằng `assert`, và test ngẫu nhiên có seed trong ràng buộc.

In [22]:
sol = Solution()

# 1) Ví dụ đề
assert sol.findReplaceString("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"
assert sol.findReplaceString("abcd", [0, 2], ["ab", "ec"], ["eee", "ffff"]) == "eeecd"
print("ví dụ đề OK")

# 2) Edge case
assert sol.findReplaceString("a", [0], ["a"], ["b"]) == "b"                      # chuỗi một ký tự
assert sol.findReplaceString("a", [0], ["b"], ["c"]) == "a"                      # không khớp
assert sol.findReplaceString("abcd", [3, 0], ["d", "a"], ["DD", "A"]) == "AbcDD"  # chỉ số không sắp xếp
assert sol.findReplaceString("abcd", [0], ["abcd"], ["x"]) == "x"                # thay toàn bộ chuỗi
assert sol.findReplaceString("abcd", [0], ["abcde"], ["x"]) == "abcd"            # nguồn dài hơn chuỗi: không khớp
assert sol.findReplaceString("abcd", [2], ["cde"], ["x"]) == "abcd"              # nguồn vượt quá cuối chuỗi
assert sol.findReplaceString("abcd", [1], ["bc"], ["bc"]) == "abcd"              # target bằng source
assert sol.findReplaceString("aaaa", [1], ["a"], ["bb"]) == "abbaa"              # chỉ thay đúng vị trí 1
assert sol.findReplaceString("aaaa", [0, 2], ["aa", "aa"], ["b", "c"]) == "bc"   # hai phép liền kề không chồng
assert sol.findReplaceString("abcd", [1, 1], ["bc", "bx"], ["Z", "W"]) == "aZd"  # cùng chỉ số: chỉ một khớp
assert sol.findReplaceString("abcd", [1, 1], ["bx", "bc"], ["W", "Z"]) == "aZd"  # đổi thứ tự hai phép
assert sol.findReplaceString("abcdef", [4, 0, 2], ["ef", "ab", "cd"], ["1", "2", "3"]) == "231"
print("edge case OK")

ví dụ đề OK
edge case OK


In [23]:
# 3) Phản ví dụ cho các lỗi (xem mục 13)
def wrong_sequential(s, indices, sources, targets):
    res = s
    for i, src, tgt in zip(indices, sources, targets):                      # SAI: thay lần lượt trên chuỗi đã sửa
        if res.startswith(src, i):
            res = res[:i] + tgt + res[i + len(src):]
    return res

assert wrong_sequential("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebcd" != "eeebffff"
print("lỗi 'thay lần lượt' bị bắt bởi Ví dụ 1")

# 4) Đối chiếu 5 cách + oracle theo ô trên nhiều input ngẫu nhiên (chỉ số không sắp xếp)
funcs = [find_replace_right_to_left, find_replace_marker, find_replace_sorted, find_replace_offset]
classes = [SolutionMarker(), SolutionSorted(), SolutionOffset(), Solution()]
rng = random.Random(2026)
for _ in range(2500):
    s, idx, src, tgt = random_case(rng, max_n=rng.choice([5, 20, 60]), alphabet=rng.choice(["a", "ab", "abc"]))
    expected = oracle_slots(s, idx, src, tgt)
    for f in funcs:
        assert f(s, idx, src, tgt) == expected, (f.__name__, s, idx, src, tgt)
    for c in classes:
        assert c.findReplaceString(s, idx, src, tgt) == expected, (type(c).__name__, s, idx, src, tgt)
print("2500 input ngẫu nhiên: mọi cách khớp oracle")

lỗi 'thay lần lượt' bị bắt bởi Ví dụ 1


2500 input ngẫu nhiên: mọi cách khớp oracle


In [24]:
# 5) Test lớn đúng ràng buộc (n = 1000, k = 100) và tính chất
rng = random.Random(5)
for _ in range(10):
    s, idx, src, tgt = max_case(rng)
    expected = oracle_slots(s, idx, src, tgt)
    for c in classes:
        assert c.findReplaceString(s, idx, src, tgt) == expected
print("n = 1000, k = 100: 10 test OK")

# 6) Tính chất: độ dài kết quả = n + tổng (len(target) - len(source)) trên các phép khớp
rng = random.Random(77)
for _ in range(1500):
    s, idx, src, tgt = random_case(rng)
    res = sol.findReplaceString(s, idx, src, tgt)
    delta = sum(len(t) - len(c) for i, c, t in zip(idx, src, tgt) if s.startswith(c, i))
    assert len(res) == len(s) + delta
# không có phép nào khớp -> kết quả bằng s
for _ in range(300):
    s, idx, src, tgt = random_case(rng, match_prob=0.0, alphabet="ab")
    if not any(s.startswith(c, i) for i, c in zip(idx, src)):
        assert sol.findReplaceString(s, idx, src, tgt) == s
print("test tính chất OK")

n = 1000, k = 100: 10 test OK


test tính chất OK

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Thay lần lượt theo thứ tự đầu vào trên chuỗi đang bị sửa

Đây là lỗi **phổ biến nhất**: sau phép đầu tiên, chuỗi đổi độ dài nên các chỉ số sau đó bị lệch.

In [25]:
def wrong_sequential_in_order(s, indices, sources, targets):
    res = s
    for i, src, tgt in zip(indices, sources, targets):
        if res.startswith(src, i):                                   # SAI: res đã bị sửa
            res = res[:i] + tgt + res[i + len(src):]
    return res

print("sai :", wrong_sequential_in_order("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]), "(kỳ vọng eeebffff)")
print("đúng:", Solution().findReplaceString("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))

sai : eeebcd (kỳ vọng eeebffff)
đúng: eeebffff


**Vì sao sai:** sau khi thay `"a"` bằng `"eee"`, chuỗi là `"eeebcd"`; chỉ số 2 giờ chứa `"e"` chứ không phải `"cd"` nên phép thứ hai bị bỏ qua nhầm. **Cách tránh:** so khớp trên `s` **gốc**, hoặc xử lý từ phải sang trái, hoặc dùng `offset`.

### Lỗi 2 — Quên nhảy qua đoạn nguồn khi dựng kết quả

Ghi `target` nhưng vẫn `p += 1`, nên các ký tự còn lại của `source` bị ghi thêm.

In [26]:
def wrong_no_skip(s, indices, sources, targets):
    n = len(s)
    d = [-1] * n
    for k, (i, src) in enumerate(zip(indices, sources)):
        if s.startswith(src, i):
            d[i] = k
    out = []
    p = 0
    while p < n:
        if d[p] != -1:
            out.append(targets[d[p]])
            p += 1                                       # SAI: phải p += len(sources[d[p]])
        else:
            out.append(s[p])
            p += 1
    return "".join(out)

print("sai :", wrong_no_skip("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]), "(kỳ vọng eeebffff)")
print("đúng:", Solution().findReplaceString("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))

sai : eeebffffd (kỳ vọng eeebffff)
đúng: eeebffff


**Vì sao sai:** sau khi thay `"cd"` ta vẫn tiếp tục ở chỉ số 3 và ghi thêm `"d"`. **Quy tắc:** thay xong phải bỏ qua **độ dài của source**, không phải 1.

### Lỗi 3 — Dùng `s.replace(src, tgt)`

`str.replace` thay **mọi** chỗ xuất hiện của `src` và hoàn toàn **không dùng chỉ số**.

In [27]:
def wrong_replace(s, indices, sources, targets):
    for src, tgt in zip(sources, targets):
        s = s.replace(src, tgt)                          # SAI
    return s

print("sai :", wrong_replace("aaaa", [1], ["a"], ["b"]), "(kỳ vọng abaa)")
print("đúng:", Solution().findReplaceString("aaaa", [1], ["a"], ["b"]))

sai : bbbb (kỳ vọng abaa)
đúng: abaa


**Vì sao sai:** đề chỉ thay đoạn **tại đúng chỉ số** đã cho, không thay các chỗ khác trùng nội dung.

### Lỗi 4 — Kiểm tra khớp bằng `src in s[i:]` (khớp ở bất kỳ đâu)

`src in s[i:]` chỉ nói rằng `src` xuất hiện **đâu đó** từ vị trí `i` trở đi, không phải **bắt đầu đúng tại** `i`.

In [28]:
def wrong_in_check(s, indices, sources, targets):
    n = len(s)
    d = [-1] * n
    for k, (i, src) in enumerate(zip(indices, sources)):
        if src in s[i:]:                                 # SAI: khớp ở bất kỳ đâu sau i
            d[i] = k
    out = []
    p = 0
    while p < n:
        if d[p] != -1:
            out.append(targets[d[p]])
            p += len(sources[d[p]])
        else:
            out.append(s[p])
            p += 1
    return "".join(out)

print("sai :", wrong_in_check("abcd", [0], ["cd"], ["X"]), "(kỳ vọng abcd)")
print("đúng:", Solution().findReplaceString("abcd", [0], ["cd"], ["X"]))

sai : Xcd (kỳ vọng abcd)
đúng: abcd


**Vì sao sai:** `"cd"` có mặt trong `"abcd"` nhưng không bắt đầu tại chỉ số 0. Dùng `s.startswith(src, i)` hoặc `s[i:i + len(src)] == src`.

### Lỗi 5 — Ghép theo con trỏ nhưng quên sắp xếp `indices`

Cách 3 yêu cầu duyệt các phép theo thứ tự **tăng dần** của chỉ số. Đầu vào có thể không sắp xếp.

In [29]:
def wrong_no_sort(s, indices, sources, targets):
    parts = []
    prev = 0
    for i, src, tgt in zip(indices, sources, targets):   # SAI: không sắp xếp
        if s.startswith(src, i):
            parts.append(s[prev:i])
            parts.append(tgt)
            prev = i + len(src)
    parts.append(s[prev:])
    return "".join(parts)

print("sai :", wrong_no_sort("abcd", [2, 0], ["cd", "a"], ["ffff", "eee"]), "(kỳ vọng eeebffff)")
print("đúng:", Solution().findReplaceString("abcd", [2, 0], ["cd", "a"], ["ffff", "eee"]))

sai : abffffeeebcd (kỳ vọng eeebffff)
đúng: eeebffff


**Vì sao sai:** sau phép `(2, "cd")`, `prev = 4`; phép `(0, "a")` lấy `s[4:0] = ""` rồi đặt `prev = 1`, ghép `s[1:] = "bcd"` ở cuối — kết quả lặp phần đã thay. **Quy tắc:** muốn ghép bằng con trỏ thì phải sắp xếp theo chỉ số trước (hoặc dùng mảng `d` như Cách 2, không cần sắp xếp).

## 14. Mẹo & nhận dạng pattern

- **"Thay thế đồng thời" / "dựa trên chuỗi gốc"** → đừng sửa chuỗi trong lúc quét. Tách thành **hai pha**: *thu thập quyết định* (chỉ đọc) rồi *dựng kết quả* một lần.
- **Dịch chỉ số** khi sửa chuỗi: xử lý từ **phải sang trái**, hoặc tính **offset**, hoặc dựng chuỗi **mới** theo chuỗi gốc (tránh sửa tại chỗ).
- **`startswith(prefix, start)`** là công cụ tốt nhất để hỏi "`src` có bắt đầu đúng tại `i` không".
- **Mảng đánh dấu theo vị trí** (`d[i] = k`): cách phổ biến để "ghi chú" các sự kiện bắt đầu tại từng vị trí rồi quét một lượt.
- **Ghép mảnh bằng `"".join`** thay vì `+=` nhiều lần.
- Bài họ hàng: Replace All Occurrences in String (3 test), String Compression (443), Encode and Decode, Add Bold Tag (616) và Merge Intervals (56) về tư duy "các đoạn rời nhau", Replace Words (648).

## 15. Biến thể

### 15.1. Cho phép chồng lấn: phép đứng trước trong đầu vào được ưu tiên

Nếu các đoạn có thể chồng nhau, cần quy ước. Giả sử: duyệt các phép theo thứ tự đầu vào; phép nào **khớp** và **không chạm** các vị trí đã bị phép trước chiếm thì được áp dụng, ngược lại bỏ qua. Ta đánh dấu vị trí đã chiếm bằng mảng `taken`. Đây là Bài 5 ở mục 17.

In [30]:
def find_replace_overlap_priority(s, indices, sources, targets):
    n = len(s)
    taken = [False] * n
    chosen = {}                                         # chỉ số bắt đầu -> (độ dài nguồn, target)
    for i, src, tgt in zip(indices, sources, targets):
        if s.startswith(src, i) and not any(taken[i:i + len(src)]):
            for j in range(i, i + len(src)):
                taken[j] = True
            chosen[i] = (len(src), tgt)
    parts = []
    p = 0
    while p < n:
        if p in chosen:
            length, tgt = chosen[p]
            parts.append(tgt)
            p += length
        else:
            parts.append(s[p])
            p += 1
    return "".join(parts)

# "ab" ở 0 và "bc" ở 1 chồng nhau; phép đứng trước ("ab") thắng
print(find_replace_overlap_priority("abc", [0, 1], ["ab", "bc"], ["X", "Y"]))   # Xc
print(find_replace_overlap_priority("abc", [1, 0], ["bc", "ab"], ["Y", "X"]))   # aY
# khi không chồng nhau thì giống Solution
print(find_replace_overlap_priority("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))   # eeebffff

Xc
aY
eeebffff


### 15.2. Ngữ nghĩa **tuần tự** (không đồng thời)

Nếu mỗi phép áp dụng lên chuỗi **vừa được sửa** bởi phép trước (theo thứ tự đầu vào), kết quả khác — và chỉ số có thể "trượt" vào nơi khác. Đây chính xác là hàm `wrong_sequential` ở mục 13: nó **không sai** với đề *tuần tự*, chỉ sai với đề *đồng thời*. Đối chiếu để thấy bài này khác biệt ở đâu:

In [31]:
def apply_sequentially(s, indices, sources, targets):
    res = s
    for i, src, tgt in zip(indices, sources, targets):
        if res.startswith(src, i):
            res = res[:i] + tgt + res[i + len(src):]
    return res

print("đồng thời:", Solution().findReplaceString("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))
print("tuần tự  :", apply_sequentially("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]))
# Nếu thay từ phải sang trái và các đoạn rời nhau thì hai ngữ nghĩa trùng nhau:
print("tuần tự, chỉ số giảm dần:", apply_sequentially("abcd", [2, 0], ["cd", "a"], ["ffff", "eee"]))

đồng thời: eeebffff
tuần tự  : eeebcd
tuần tự, chỉ số giảm dần: eeebffff


### 15.3. Chỉ cần biết độ dài kết quả / số phép có hiệu lực

Độ dài kết quả `= n + Σ (len(target) − len(source))` trên các phép **khớp**; không cần dựng chuỗi. Dùng một lượt qua các phép, `O(L_s)`.

In [32]:
def result_length_and_count(s, indices, sources, targets):
    length = len(s)
    effective = 0
    for i, src, tgt in zip(indices, sources, targets):
        if s.startswith(src, i):
            length += len(tgt) - len(src)
            effective += 1
    return length, effective

for s_, idx, src, tgt, expected in examples:
    L, c = result_length_and_count(s_, idx, src, tgt)
    assert L == len(expected)
    print(s_, "->", expected, "| độ dài", L, "| số phép có hiệu lực", c)

abcd -> eeebffff | độ dài 8 | số phép có hiệu lực 2
abcd -> eeecd | độ dài 5 | số phép có hiệu lực 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thay từ phải sang trái | Sắp xếp giảm dần, sửa chuỗi tại chỗ (không dịch chỉ số bên trái) | `O(k log k + k·n)` | `O(n + L)` | Ngắn gọn; ý "tránh dịch chỉ số" |
| 2. Mảng đánh dấu + quét | `d[i] = k` rồi dựng một lượt, nhảy qua nguồn | `O(n + L)` | `O(n + L)` | **Lời giải chuẩn nộp LeetCode**; không cần sắp xếp |
| 3. Sắp xếp + con trỏ `prev` | Sắp theo chỉ số, ghép mảnh giữa các phép | `O(k log k + n + L)` | `O(n + L)` | Khi `k` nhỏ so với `n`; không cần mảng dài `n` |
| 4. Sắp xếp + `offset` | So khớp trên gốc, ánh xạ chỉ số bằng độ lệch | `O(k log k + k·(n + L))` | `O(n + L)` | Khi phải sửa trực tiếp một chuỗi/buffer có sẵn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(Solution().findReplaceString("abcde", [1], ["bc"], ["XYZ"]))
print(Solution().findReplaceString("abcde", [1], ["bd"], ["XYZ"]))
print(Solution().findReplaceString("aaaa", [0, 2], ["a", "a"], ["b", "c"]))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "abcde"`, `indices = [3, 1]`, `sources = ["de", "bc"]`, `targets = ["YZ", "X"]` (chỉ số **không sắp xếp**): hãy tính tay mảng đánh dấu `d` (độ dài 5; `d[i]` là số thứ tự phép có hiệu lực bắt đầu tại `i`, hoặc `-1`) và chuỗi kết quả. Điền vào các biến.

In [34]:
# Bài 2: điền kết quả tính tay
hand_d = None        # ví dụ: [-1, -1, -1, -1, -1]
hand_result = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành Cách 2 (mảng đánh dấu).

In [35]:
def exercise_3_find_replace(s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
    n = len(s)
    # TODO 1: d = [-1] * n
    # TODO 2: với mỗi phép k (dùng enumerate(zip(indices, sources))): nếu s.startswith(src, i) thì d[i] = k
    # TODO 3: duyệt p từ 0 đến n-1 với danh sách ans:
    #     nếu d[p] != -1: ans.append(targets[d[p]]); p += len(sources[d[p]])
    #     ngược lại: ans.append(s[p]); p += 1
    # TODO 4: return "".join(ans)
    return None


# assert exercise_3_find_replace("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"
# assert exercise_3_find_replace("abcd", [0, 2], ["ab", "ec"], ["eee", "ffff"]) == "eeecd"
# assert exercise_3_find_replace("abcd", [2, 0], ["cd", "a"], ["ffff", "eee"]) == "eeebffff"

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_sorted(s, indices, sources, targets)`: sắp xếp các phép theo chỉ số, dùng con trỏ `prev` để ghép các mảnh.

In [36]:
def exercise_4_sorted(s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
    # TODO
    pass


# assert exercise_4_sorted("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"
# assert exercise_4_sorted("abcdefgh", [5, 1, 3], ["fg", "bc", "zz"], ["X", "YYY", "Q"]) == "aYYYdeXh"
# assert exercise_4_sorted("abc", [0], ["abc"], ["x"]) == "x"

### Bài 5 — Mini challenge: các phép có thể chồng lấn (khó)

Viết `exercise_5_overlap(s, indices, sources, targets)` cho phiên bản **cho phép chồng lấn**: duyệt các phép theo thứ tự đầu vào; phép nào **khớp** trên `s` gốc và **không chạm** bất kỳ vị trí nào đã bị phép trước (đã được áp dụng) chiếm thì được áp dụng; phép khác bị bỏ qua. Trả về chuỗi kết quả.

```python
exercise_5_overlap("abc", [0, 1], ["ab", "bc"], ["X", "Y"])  -> "Xc"
exercise_5_overlap("abc", [1, 0], ["bc", "ab"], ["Y", "X"])  -> "aY"
exercise_5_overlap("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) -> "eeebffff"
```

In [37]:
def exercise_5_overlap(s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
    # TODO
    pass


# assert exercise_5_overlap("abc", [0, 1], ["ab", "bc"], ["X", "Y"]) == "Xc"
# assert exercise_5_overlap("abc", [1, 0], ["bc", "ab"], ["Y", "X"]) == "aY"
# assert exercise_5_overlap("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Mọi phép đều phải được **kiểm tra trên chuỗi gốc** `s`, không phải trên chuỗi đang bị sửa.
- *Mức 2 (vừa):* Tách hai pha: pha 1 xác định phép nào có hiệu lực (`s.startswith(src, i)`); pha 2 dựng kết quả từ trái sang, nhảy qua độ dài `source` sau khi ghi `target`.
- *Mức 3 (gần đáp án):* Cách 2: `d[i] = k` khi khớp; quét `p`: `if d[p] != -1: ans.append(targets[d[p]]); p += len(sources[d[p]]) else: ans.append(s[p]); p += 1`. Cách 3: `for i, src, tgt in sorted(zip(...), key=lambda op: op[0])`: nếu khớp thì `parts.append(s[prev:i])`, `parts.append(tgt)`, `prev = i + len(src)`; cuối cùng `parts.append(s[prev:])`.

**Bài 5**

- *Mức 1:* Cần nhớ vị trí nào đã bị chiếm: một mảng `taken` độ dài `n`.
- *Mức 2:* Với từng phép theo thứ tự đầu vào: khớp trên `s` gốc **và** `not any(taken[i:i + len(src)])` thì chọn; đánh dấu `taken` cho đoạn đó và nhớ `(độ dài, target)` tại chỉ số `i`.
- *Mức 3:* Dựng kết quả bằng cách quét `p`: nếu `p` là đầu một đoạn được chọn thì ghi `target` và nhảy qua độ dài nguồn; ngược lại ghi `s[p]`. (Vì các đoạn được chọn không chồng nhau nên cách quét này đúng.)

## 19. Đáp án tham khảo

### Đáp án Bài 1

- `findReplaceString("abcde", [1], ["bc"], ["XYZ"])` → `"aXYZde"`.
- `findReplaceString("abcde", [1], ["bd"], ["XYZ"])` → `"abcde"` (`"bd"` không khớp tại chỉ số 1: đó là `"bc"`).
- `findReplaceString("aaaa", [0, 2], ["a", "a"], ["b", "c"])` → `"baca"`: chỉ số 0 thay `"a"` bằng `"b"`, chỉ số 2 thay `"a"` bằng `"c"`, còn chỉ số 1 và 3 giữ `"a"`.

In [38]:
assert Solution().findReplaceString("abcde", [1], ["bc"], ["XYZ"]) == "aXYZde"
assert Solution().findReplaceString("abcde", [1], ["bd"], ["XYZ"]) == "abcde"
assert Solution().findReplaceString("aaaa", [0, 2], ["a", "a"], ["b", "c"]) == "baca"
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

Kiểm tra trên `s = "abcde"`: phép 0 `(3, "de", "YZ")`: `s[3:5] = "de"` ✓ → `d[3] = 0`. Phép 1 `(1, "bc", "X")`: `s[1:3] = "bc"` ✓ → `d[1] = 1`. Vậy `d = [-1, 1, -1, 0, -1]`.

Dựng: `p=0`: ghi `a`; `p=1`: `d[1]=1` → ghi `"X"`, nhảy 2 → `p=3`; `d[3]=0` → ghi `"YZ"`, nhảy 2 → `p=5`. Kết quả `"aXYZ"`.

In [39]:
hand_d = [-1, 1, -1, 0, -1]
hand_result = "aXYZ"
assert hand_d == build_marker("abcde", [3, 1], ["de", "bc"])
assert hand_result == Solution().findReplaceString("abcde", [3, 1], ["de", "bc"], ["YZ", "X"])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_solution(s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
    n = len(s)
    d = [-1] * n
    for k, (i, src) in enumerate(zip(indices, sources)):
        if s.startswith(src, i):
            d[i] = k
    ans = []
    p = 0
    while p < n:
        if d[p] != -1:
            ans.append(targets[d[p]])
            p += len(sources[d[p]])
        else:
            ans.append(s[p])
            p += 1
    return "".join(ans)

assert exercise_3_solution("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"
assert exercise_3_solution("abcd", [0, 2], ["ab", "ec"], ["eee", "ffff"]) == "eeecd"
assert exercise_3_solution("abcd", [2, 0], ["cd", "a"], ["ffff", "eee"]) == "eeebffff"
rng = random.Random(3)
for _ in range(300):
    s, idx, src, tgt = random_case(rng)
    assert exercise_3_solution(s, idx, src, tgt) == oracle_slots(s, idx, src, tgt)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_solution(s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
    parts = []
    prev = 0
    for i, src, tgt in sorted(zip(indices, sources, targets), key=lambda op: op[0]):
        if s.startswith(src, i):
            parts.append(s[prev:i])
            parts.append(tgt)
            prev = i + len(src)
    parts.append(s[prev:])
    return "".join(parts)

assert exercise_4_solution("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"
assert exercise_4_solution("abcdefgh", [5, 1, 3], ["fg", "bc", "zz"], ["X", "YYY", "Q"]) == "aYYYdeXh"
assert exercise_4_solution("abc", [0], ["abc"], ["x"]) == "x"
rng = random.Random(4)
for _ in range(300):
    s, idx, src, tgt = random_case(rng)
    assert exercise_4_solution(s, idx, src, tgt) == oracle_slots(s, idx, src, tgt)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_solution(s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
    n = len(s)
    taken = [False] * n
    chosen = {}
    for i, src, tgt in zip(indices, sources, targets):
        if s.startswith(src, i) and not any(taken[i:i + len(src)]):
            for j in range(i, i + len(src)):
                taken[j] = True
            chosen[i] = (len(src), tgt)
    parts = []
    p = 0
    while p < n:
        if p in chosen:
            length, tgt = chosen[p]
            parts.append(tgt)
            p += length
        else:
            parts.append(s[p])
            p += 1
    return "".join(parts)

assert exercise_5_solution("abc", [0, 1], ["ab", "bc"], ["X", "Y"]) == "Xc"
assert exercise_5_solution("abc", [1, 0], ["bc", "ab"], ["Y", "X"]) == "aY"
assert exercise_5_solution("abcd", [0, 2], ["a", "cd"], ["eee", "ffff"]) == "eeebffff"

# khi đầu vào KHÔNG chồng lấn thì trùng Solution; và đối chiếu với một cách làm khác (oracle theo ô, có ưu tiên)
def oracle_overlap(s, indices, sources, targets):
    slots = list(s)
    used = set()
    for i, src, tgt in zip(indices, sources, targets):
        rng_ = set(range(i, i + len(src)))
        if s[i:i + len(src)] == src and not (rng_ & used):
            used |= rng_
            slots[i] = tgt
            for j in range(i + 1, i + len(src)):
                slots[j] = ""
    return "".join(slots)

rng = random.Random(55)
for _ in range(500):
    s, idx, src, tgt = random_case(rng)
    assert exercise_5_solution(s, idx, src, tgt) == Solution().findReplaceString(s, idx, src, tgt)
for _ in range(1500):                                  # đầu vào có thể chồng lấn
    n = rng.randint(1, 12)
    s = "".join(rng.choice("ab") for _ in range(n))
    k = rng.randint(1, 5)
    idx = [rng.randrange(n) for _ in range(k)]
    src = [s[i:i + rng.randint(1, 3)] if rng.random() < 0.8 else "b" for i in idx]
    tgt = ["".join(rng.choice("xy") for _ in range(rng.randint(1, 3))) for _ in range(k)]
    assert exercise_5_solution(s, idx, src, tgt) == oracle_overlap(s, idx, src, tgt)
print("Bài 5 OK")

Bài 5 OK

## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def findReplaceString(self, s: str, indices: List[int], sources: List[str], targets: List[str]) -> str:
        n = len(s)
        d = [-1] * n
        for k, (i, src) in enumerate(zip(indices, sources)):
            if s.startswith(src, i):            # so khớp trên chuỗi GỐC
                d[i] = k
        ans = []
        p = 0
        while p < n:
            if d[p] != -1:
                ans.append(targets[d[p]])
                p += len(sources[d[p]])         # nhảy qua đoạn nguồn
            else:
                ans.append(s[p])
                p += 1
        return "".join(ans)
```

Câu thần chú: **"Hỏi trên chuỗi gốc, ghi vào mảng đánh dấu; rồi quét một lần: gặp phép thì ghi target và nhảy qua source."**

### Checklist trước khi submit

- [ ] Mọi so khớp làm trên `s` **gốc** (đồng thời), không sửa `s` khi đang kiểm tra.
- [ ] Dùng `s.startswith(src, i)` (đúng tại vị trí `i`), không dùng `in`.
- [ ] Sau khi ghi `target`, nhảy `len(source)` ký tự trong chuỗi gốc.
- [ ] Không giả định `indices` đã sắp xếp (dùng mảng `d`, hoặc tự `sorted`).
- [ ] Ghép bằng danh sách + `"".join`.
- [ ] Đã thử: không phép nào khớp, nguồn vượt quá cuối chuỗi, thay cả chuỗi, `target` dài/ngắn hơn `source`, chỉ số lộn xộn, hai phép liền kề.
- [ ] Đã đối chiếu oracle theo ô trên input ngẫu nhiên.
- [ ] Skeleton đúng: `class Solution`, method `findReplaceString(self, s, indices, sources, targets) -> str`.